In [1]:
import modules.data as d
import modules.model as m
import modules.train as t
import modules.utils as u

import torch.nn as nn
from pathlib import Path

In [2]:
# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device()

# get data
data = d.Data(
    # datasets
    tcga_project = 'TCGA-BRCA',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',

    # dirs
    tcga_dir = datasets / 'tcga',
    relation_filepath = datasets / 'other/relation_ohe.csv',
    
    # col, filter
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Metastatic']},
    max_subset=120,
)

*** Device() ***
device = cuda:4

**** Data() ****
log0_method      log1p            str
class_weights    (6,)             Tensor (cuda:4)
gene_counts      (4383, 567)      DataFrame
metadata         (567, 3)         DataFrame
relation         (32798, 18)      DataFrame
node_id_map      4383             dict
masks            305              list
X                (567, 4383, 1)   Tensor (cuda:4)
y                (567, 6)         Tensor (cuda:4)
y_labels         6                list
num_samples      567              int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int



---

In [3]:
import pandas as pd
from typing import Literal

from pandas import DataFrame
from torch import Tensor
from modules.model import MLP, GCN

In [4]:
class MLPAutoencoder(nn.Module):
    def __init__(self, in_features:int, embedding_size:int=8, encoder_kwargs:dict={}, decoder_kwargs:dict={}, squeeze:bool=False, unsqueeze:bool=False):
        super().__init__()

        # assign instance variables
        self.squeeze = squeeze
        self.unsqueeze = unsqueeze
        
        # define layers
        self.encoder = MLP(
            in_features=in_features,
            out_features=embedding_size,
            **encoder_kwargs
        )
        self.decoder = MLP(
            in_features=embedding_size,
            out_features=in_features,
            **decoder_kwargs
        )
    
    def encode(self, X:Tensor, squeeze:bool=False):
        # squeeze if applicable
        if squeeze == True:
            X = X.squeeze(-1)

        # encode
        z = self.encoder(X)

        return z
    
    def decode(self, z:Tensor, unsqueeze:bool=False):
        # decode
        X_hat = self.decoder(z)

        # unsqueeze if applicable
        if unsqueeze == True:
            X_hat = X_hat.unsqueeze(-1)

        return X_hat
    
    def forward(self, X:Tensor):
        # encode, decode
        z = self.encode(X, self.squeeze)
        X_hat = self.decode(z, self.unsqueeze)

        return X_hat
        


In [5]:
mlp = MLPAutoencoder(
    in_features=data.num_nodes,
    embedding_size=8,
    squeeze=True,
    unsqueeze=True
)

mlp(data.X).shape

torch.Size([567, 4383, 1])

---

In [6]:
class GCNAutoencoder(nn.Module):
    def __init__(
            self, in_features:int, embedding_size:int, relation:DataFrame, num_nodes:int, 
            gcn_kwargs:dict={}, encoder_kwargs:dict={}, decoder_kwargs:dict={},
            adj_out:bool=False, adj_in:bool=False, adj_self:bool=True, bias:bool=True, normalize:Literal['sym','row']='row'
        ):
        super().__init__()

        # set gcn kwargs
        gcn_kwargs = {
            'adj_out':adj_out, 
            'adj_in':adj_in,
            'adj_self':adj_self,
            'bias':bias,
            'normalize':normalize,
            **gcn_kwargs
        }

        # define layers
        self.gcn = GCN(in_features=in_features, out_features=1, relation=relation, num_nodes=num_nodes, **gcn_kwargs)
        self.encoder = MLP(in_features=num_nodes, out_features=embedding_size, **encoder_kwargs)   
        self.decoder = MLP(in_features=embedding_size, out_features=num_nodes, **decoder_kwargs)

    def encode(self, X:Tensor):
        # X: (batch_size, num_nodes, num_features)

        # node embedding >> (batch_size, num_nodes, 1)
        H = self.gcn(X)

        # squeeze >> (batch_size, num_nodes)
        H = H.squeeze(-1)

        # encode >> (batch_size, embedding_size)
        z = self.encoder(H)

        return z
    
    def decode(self, z:Tensor):
        # z: (batch_size, embedding_size)

        # decode >> (batch_size, num_nodes)
        X_hat = self.decoder(z)

        # unqueeze >> (batch_size, num_nodes, 1)
        X_hat = X_hat.unsqueeze(-1)

        return X_hat

    def forward(self, X:Tensor):
        # encode, decode
        z = self.encode(X)
        X_hat = self.decode(z)

        return X_hat


In [7]:
gcn = GCNAutoencoder(
    in_features=data.num_features,
    embedding_size=8,
    relation=data.relation,
    num_nodes=data.num_nodes,
)

gcn(data.X).shape

torch.Size([567, 4383, 1])

---

In [4]:
import torch
import torch.nn.functional as F
from modules.model import _get_adj

In [6]:
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features:int, out_features:int, relation:DataFrame, num_nodes:int):
        super().__init__()
        # inst vars
        self.out_features=out_features
        self.num_nodes=num_nodes
        self.leakyrelu = nn.LeakyReLU(0.2)

        # init weight 'W'
        self.weight = nn.Parameter(torch.empty(in_features, self.out_features))
        nn.init.xavier_uniform_(self.weight.data, gain=1.414)

        # init weight 'a'
        self.a = nn.Parameter(torch.empty(size=(2*self.out_features, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.414)

        # get adj
        self.adj = _get_adj(relation, self.num_nodes)

    def _get_attention(self, H:Tensor):
        # get [h_i||h_j]
        h_i = H.unsqueeze(2).repeat(1, 1, self.num_nodes, 1)  # (batch_size, N, N, out_features)
        h_j = H.unsqueeze(1).repeat(1, self.num_nodes, 1, 1)  # (batch_size, N, N, out_features)
        h_ij = torch.cat([h_i, h_j], dim=-1) # (batch_size, N, N, 2*out_features)

        # get raw attention
        e = self.leakyrelu(torch.matmul(h_ij, self.a).squeeze(-1)) # (batch_size, N, N, 1) >> squeeze (batch_size, N, N)

        # mask, normalize e
        zero_mask = -9e15 * torch.ones_like(e)
        attention = torch.where(self.adj > 0, e, zero_mask) # mask where adj <= 0
        attention = F.softmax(attention, dim=1) # softmax norm

        return attention
    
    def forward(self, X:Tensor):
        # node embedding
        H = torch.matmul(X, self.weight)

        # get attention
        attention = self._get_attention(H)

        # aggregate neighbor embeddings
        H = torch.matmul(attention, H)

        # activation (ELU)
        H = F.elu(H)

        return H

In [9]:
gat_layer = GraphAttentionLayer(
    in_features=data.num_features,
    out_features=8,
    relation=data.relation,
    num_nodes=data.num_nodes
)

## 44GB VRAM LMAO
gat_layer(data.X[:16,:,:]).shape

torch.Size([16, 4383, 8])

---

In [5]:
def _get_edge_index(relation:DataFrame, method:Literal['out','in']='out', source:str='idx1', target:str='idx2'):
    # get edge information >> (num_edges, 2)
    if method == 'out':
        edge_index = relation[[source, target]]
    elif method == 'in':
        edge_index = relation[[target, source]]
    
    # transpose, convert to tensor >> (2, num_edges) 
    edge_index = Tensor(edge_index.values.T).int()

    return edge_index

In [22]:
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features:int, out_features:int, relation:DataFrame, num_nodes:int):
        super().__init__()

        # inst vars
        self.out_features = out_features
        self.num_nodes = num_nodes
        self.leakyrelu = nn.LeakyReLU(0.2)

        # init weight 'W'
        self.weight = nn.Parameter(torch.empty(in_features, self.out_features))
        nn.init.xavier_uniform_(self.weight.data, gain=1.414)

        # init weight 'a'
        self.a = nn.Parameter(torch.empty(size=(2*self.out_features, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.414)

        # get edge indices
        self.edge_index = _get_edge_index(relation, method='in')

    def forward(self, X:Tensor):
        H = torch.matmul(X, self.weight)  # (batch_size, num_nodes, out_features)

        # Efficiently compute attention coefficients
        edge_h = torch.cat((H[:, self.edge_index[0, :]], H[:, self.edge_index[1, :]]), dim=-1)  # (batch_size, num_edges, 2*out_features)
        e = self.leakyrelu(torch.matmul(edge_h, self.a).squeeze(-1))  # (batch_size, num_edges)

        # Sparse attention using scatter
        attention = torch.zeros(X.size(0), self.num_nodes, self.num_nodes, device=X.device).fill_(-9e15)
        attention[:, self.edge_index[0], self.edge_index[1]] = e

        # Softmax norm
        attention = F.softmax(attention, dim=-1)

        # Aggregate neighbor embeddings efficiently
        H_prime = torch.matmul(attention, H)

        return H_prime

In [23]:
gat_layer = GraphAttentionLayer(
    in_features=data.num_features,
    out_features=8,
    relation=data.relation,
    num_nodes=data.num_nodes
)

## ONLY 3GB?? WTF??
gat_layer(data.X[:16,:,:]).shape

torch.Size([16, 4383, 8])

---

In [14]:
from modules.model import _get_layers

In [28]:
class GAT(nn.Module):
    def __init__(self, in_features:int, out_features:int, relation:DataFrame, num_nodes:int, hidden_dims:list=[], act_fn=nn.ELU(), end_fn=nn.ELU()):
        super().__init__()

        # define layers
        layers = _get_layers(
            in_features=in_features,
            out_features=out_features,
            layer_class=GraphAttentionLayer,
            layer_kwargs={
                'relation':relation,
                'num_nodes':num_nodes,
            },
            hidden_dims=hidden_dims,
            act_fn=act_fn,
            end_fn=end_fn
        )

        # define model
        self.model = nn.Sequential(*layers)

    def forward(self, X:Tensor):
        return self.model(X)

In [32]:
gat_model = GAT(
    in_features=data.num_features,
    out_features=8,
    relation=data.relation,
    # hidden_dims=[64],
    num_nodes=data.num_nodes
)

gat_model(data.X[:16,:,:]).shape

torch.Size([16, 4383, 8])

---

In [33]:
class GATClassifier(nn.Module):
    def __init__(self, in_features:int, out_features:int, relation:DataFrame, num_nodes:int, gat_kwargs:dict={}, mlp_kwargs:dict={}):
        super().__init__()

        # define layers
        self.gat = GAT(in_features=in_features, out_features=1, relation=relation, num_nodes=num_nodes, **gat_kwargs)
        self.mlp = MLP(in_features=num_nodes, out_features=out_features, **mlp_kwargs)

    def forward(self, X):
        # node embedding: (batch_size, num_nodes, num_features) >> (batch_size, num_nodes, 1)
        H = self.gat(X).squeeze(-1) # squeeze >> (batch_size, num_nodes)
        
        # get logits
        logits = self.mlp(H)
        
        return logits

    def predict(self, X:Tensor, as_logits:bool=True):
        # transform if raw data (not logits)
        if as_logits == False:
            X = self.forward(X)

        # convert logits to prediction
        probs = torch.softmax(X, dim=1) # softmax to probs
        y_pred = torch.argmax(probs, dim=1) # argmax to most likely class
        y_pred = F.one_hot(y_pred, probs.shape[1]) # get one-hot encoding

        return y_pred

In [37]:
gat_c = GATClassifier(
    in_features=data.num_features,
    out_features=8,
    relation=data.relation,
    num_nodes=data.num_nodes
)

gat_c(data.X[:16,:,:]).shape

torch.Size([16, 8])

---

In [41]:
class GATAutoencoder(nn.Module):
    def __init__(self, in_features:int, embedding_size:int, relation:DataFrame, num_nodes:int, gat_kwargs:dict={}, encoder_kwargs:dict={}, decoder_kwargs:dict={}):
        super().__init__()

        # define layers
        self.gat = GAT(in_features=in_features, out_features=1, relation=relation, num_nodes=num_nodes, **gat_kwargs)
        self.encoder = MLP(in_features=num_nodes, out_features=embedding_size, **encoder_kwargs)   
        self.decoder = MLP(in_features=embedding_size, out_features=num_nodes, **decoder_kwargs)

    def encode(self, X:Tensor):
        # X: (batch_size, num_nodes, num_features)

        # node embedding >> (batch_size, num_nodes, 1)
        H = self.gat(X)

        # squeeze >> (batch_size, num_nodes)
        H = H.squeeze(-1)

        # encode >> (batch_size, embedding_size)
        z = self.encoder(H)

        return z
    
    def decode(self, z:Tensor):
        # z: (batch_size, embedding_size)

        # decode >> (batch_size, num_nodes)
        X_hat = self.decoder(z)

        # unqueeze >> (batch_size, num_nodes, 1)
        X_hat = X_hat.unsqueeze(-1)

        return X_hat
    
    def forward(self, X:Tensor):
        # encode, decode
        z = self.encode(X)
        X_hat = self.decode(z)

        return X_hat


In [43]:
gat_ae = GATAutoencoder(
    in_features=data.num_features,
    embedding_size=8,
    relation=data.relation,
    num_nodes=data.num_nodes
)

gat_ae(data.X[:16,:,:]).shape

torch.Size([16, 4383, 1])